# 🧽 Annotation Fixer — чистка аннотаций поверх готовых чанков

Берёт чанки из `annotate.ipynb` и **текстовым** LLM (Qwen2.5-7B, без картинок)
чистит поля: правит OCR-мусор в `caption_text`, переписывает `meaning`
аккуратнее (язык сохраняется — русский остаётся русским), ставит `flag`
(`ok` / `low_info` / `junk`). Картинки переносятся как есть, VLM не гоняется.

Устойчив к обрывам: пишет **чанками** на Drive, при перезапуске пропускает готовые.
**Запуск:** T4 GPU → `Run all`. Быстро — это текст, не картинки.


In [ ]:
# 1) Зависимости
!pip -q install "transformers>=4.44" accelerate bitsandbytes tqdm

In [ ]:
# 2) Настройки
IN_DIR     = "/content/drive/MyDrive/Memes_annotated/chunks"   # чанки из annotate.ipynb
OUT_DIR    = "/content/drive/MyDrive/Memes_fixed"              # куда писать почищенные чанки
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"   # 3B хватает для чистки текста и ~2x быстрее 7B
MAX_NEW    = 180                          # короткий вывод -> быстрее
CHUNK_SIZE = 100

In [ ]:
# 3) Смонтировать Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4) Загрузка текстового LLM (4-bit). Один раз.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto")

def chat(messages, max_new_tokens=MAX_NEW):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

print("LLM loaded on", model.device)

In [ ]:
# 5) Фиксер одной записи
import json, re

SYS = "You clean auto-generated meme annotations. Reply with ONLY one JSON object, nothing else."

def _json(s):
    m = re.search(r"\{.*\}", s, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def fix(rec):
    user = (
        "Clean the annotation fields for one meme.\n"
        "DESCRIPTION: " + rec.get("image_desc", "") + "\n"
        "OCR TEXT (may be garbled): " + rec.get("caption_text", "") + "\n"
        "DRAFT MEANING: " + rec.get("meaning", "") + "\n\n"
        "Rules (ALL output fields MUST be in ENGLISH):\n"
        "- caption_text: fix OCR errors into readable text, and TRANSLATE it to English "
        "if it is in another language (e.g. Russian). If it is pure noise or empty, return \"\".\n"
        "- meaning: a concise, accurate explanation IN ENGLISH of the joke/message, based on "
        "the description and cleaned text. If it is just an object/animal with no joke, say so briefly.\n"
        "- flag: \"ok\" normally; \"low_info\" if there is no real joke; \"junk\" if unusable.\n"
        'Reply ONLY JSON: {"caption_text": "...", "meaning": "...", "flag": "ok"}'
    )
    d = _json(chat([{"role": "system", "content": SYS}, {"role": "user", "content": user}]))
    out = dict(rec)
    if d:
        out["caption_text"] = str(d.get("caption_text", rec.get("caption_text", ""))).strip()
        out["meaning"] = str(d.get("meaning", rec.get("meaning", ""))).strip()
        out["flag"] = str(d.get("flag", "ok")).strip() or "ok"
    else:
        out["flag"] = "ok"   # не смогли распарсить — оставляем как было
    return out

In [ ]:
# 6) Резюм: готовые id берём из уже записанных ВЫХОДНЫХ чанков
import os, glob, zipfile

OUT_CHUNK_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_CHUNK_DIR, exist_ok=True)

done = set()
existing = sorted(glob.glob(os.path.join(OUT_CHUNK_DIR, "chunk_*.zip")))
for c in existing:
    try:
        with zipfile.ZipFile(c) as z:
            for line in z.read("metadata.jsonl").decode("utf-8").splitlines():
                if line.strip():
                    done.add(json.loads(line)["id"])
    except Exception as e:
        print("warn:", c, e)
out_idx = len(existing) + 1
print("уже почищено:", len(done), "| следующий выходной чанк:", out_idx)

In [ ]:
# 7) Основной цикл: читаем входные чанки -> fix -> пишем выходные чанки (резюм по id)
from tqdm.auto import tqdm

in_chunks = sorted(glob.glob(os.path.join(IN_DIR, "chunk_*.zip")))
assert in_chunks, "Во IN_DIR нет входных чанков — проверь путь."

buffer = []
preview = []
stats = {"fixed": 0, "skip_done": 0, "flags": {}}

def flush():
    global out_idx, buffer
    if not buffer:
        return
    tmp = os.path.join(OUT_CHUNK_DIR, ".tmp_chunk.zip")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        meta = []
        for h, b, rec in buffer:
            z.writestr("images/%s.webp" % h, b)
            meta.append(json.dumps(rec, ensure_ascii=False))
        z.writestr("metadata.jsonl", "\n".join(meta) + "\n")
    os.replace(tmp, os.path.join(OUT_CHUNK_DIR, "chunk_%05d.zip" % out_idx))
    out_idx += 1
    buffer = []

# посчитать всего записей для прогресс-бара
totals = []
for c in in_chunks:
    with zipfile.ZipFile(c) as z:
        totals.append(len([l for l in z.read("metadata.jsonl").decode().splitlines() if l.strip()]))
pbar = tqdm(total=sum(totals), desc="fixing")

for c in in_chunks:
    with zipfile.ZipFile(c) as z:
        metas = [json.loads(l) for l in z.read("metadata.jsonl").decode().splitlines() if l.strip()]
        for rec in metas:
            pbar.update(1)
            if rec["id"] in done:
                stats["skip_done"] += 1
                continue
            b = z.read("images/%s.webp" % rec["id"])
            fixed = fix(rec)
            done.add(rec["id"])
            fl = fixed.get("flag", "ok")
            stats["flags"][fl] = stats["flags"].get(fl, 0) + 1
            stats["fixed"] += 1
            buffer.append((rec["id"], b, fixed))
            if len(preview) < 6:
                preview.append((rec, fixed))
            if len(buffer) >= CHUNK_SIZE:
                flush()
pbar.close()
flush()
print("готово:", stats)

In [ ]:
# 8) Слить почищенные чанки в один датасет
OUT_ZIP = os.path.join(OUT_DIR, "fixed_memes.zip")
chunks = sorted(glob.glob(os.path.join(OUT_CHUNK_DIR, "chunk_*.zip")))
n_img = 0
seen_names = set()
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zout:
    meta_lines = []
    for c in chunks:
        with zipfile.ZipFile(c) as zin:
            for name in zin.namelist():
                if name.startswith("images/") and name not in seen_names:
                    zout.writestr(name, zin.read(name))
                    seen_names.add(name)
                    n_img += 1
                elif name == "metadata.jsonl":
                    meta_lines.append(zin.read(name).decode("utf-8").strip())
    zout.writestr("metadata.jsonl", "\n".join(l for l in meta_lines if l) + "\n")
print("собрано", n_img, "картинок в", OUT_ZIP)

In [ ]:
# 9) Предпросмотр «до -> после» (прогони сначала на паре чанков и оцени)
for orig, fixed in preview:
    print("OCR  :", (orig.get("caption_text", "") or "<пусто>")[:100])
    print("  ->  :", (fixed.get("caption_text", "") or "<пусто>")[:100])
    print("MEAN :", orig.get("meaning", ""))
    print("  ->  :", fixed.get("meaning", ""), "  [flag:", fixed.get("flag"), "]")
    print("-" * 70)